In [1]:
# Load the flagged listings
from configs.dataset_config import (
    DATA_DATE_SUFFIX, WITH_FEEDBACK
)

version = DATA_DATE_SUFFIX
# version = 'march_31000'

feedback = ''
if WITH_FEEDBACK:
    feedback = '_with_feedback'

check_online_status = True
listings_version = 'pts_outliers'
# listings_version = 'pts_non_sqft_outliers'
# listings_version = 'pts_and_sqft_outliers'
# listings_version = 'any_outlier'


In [2]:
"""
Check the status of outlier listings (Active vs Taken Down)

This script reads the flagged listings from the notebook output and checks
whether each outlier listing is still active or has been taken down.
"""

import pandas as pd
import requests
from tqdm import tqdm
import time

In [3]:
def check_listing_status(url):
    """
    Check if a listing URL is active or taken down.

    Args:
        url: The listing URL to check

    Returns:
        str: 'Active', 'Taken Down', or 'Error: <message>'
    """
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=10, allow_redirects=True)

        if "not-found.html" in response.url:
            return "Taken Down"
        else:
            return "Active"

    except requests.exceptions.Timeout:
        return "Error: Timeout"
    except requests.exceptions.RequestException as e:
        return f"Error: {str(e)[:50]}"
    except Exception as e:
        return f"Error: {str(e)[:50]}"


In [4]:
outliers_cols = [
        'outlier_rank',  # Add rank as first column
        'property_listing_id', 'location_lvl_0_name', 'location_id',
        'housing_type_name', 'offering_type_name','property_price_type_name', 'bedrooms', 'price', 'property_sqft', 'price_to_sqft',
        'segment_key', 'segment_count', 'relaxation_level',
        'price_median', 'property_sqft_median', 'price_to_sqft_median',
        'pts_deviation_ratio', 'price_deviation_ratio', 'sqft_deviation_ratio',  # Raw deviation ratios
        'pts_normalized_score', 'price_normalized_score', 'sqft_normalized_score',  # Normalized scores (account for relaxation)
        'is_pts_outlier', 'pts_outlier_type',
        'is_price_outlier', 'price_outlier_type',
        'is_sqft_outlier', 'sqft_outlier_type',
        'pts_multiplier', 'price_multiplier', 'sqft_multiplier',
        # 'property_permit_number',
    # 'listing_short_term_flag', 'complaince_type', 'client_id', 'salesforce_account_no'
    ]

In [5]:
input_path = f'../datasets/{version}/inference_outliers_only_{version}{feedback}.parquet'
all_listings_path = f'../datasets/{version}/inference_flagged_listings_{version}{feedback}.parquet'

print(f"Loading flagged listings from: {input_path}")
df = pd.read_parquet(input_path, columns=outliers_cols)
df['outlier_rank'] = df['outlier_rank'].astype(int)

# Create URL if not already present
if 'url' not in df.columns:
    outliers_cols.insert(1, 'url')
    df['url'] = 'http://propertyfinder.ae/to/' + df['property_listing_id']
    df = df[outliers_cols]
df

Loading flagged listings from: ../datasets/april_17000/inference_outliers_only_april_17000.parquet


,outlier_rank,url,property_listing_id,location_lvl_0_name,location_id,housing_type_name,offering_type_name,property_price_type_name,bedrooms,price,...,sqft_normalized_score,is_pts_outlier,pts_outlier_type,is_price_outlier,price_outlier_type,is_sqft_outlier,sqft_outlier_type,pts_multiplier,price_multiplier,sqft_multiplier
0,1,http://propertyfinder.ae/to/DC6YT71SJBH0WJZ1R8...,DC6YT71SJBH0WJZ1R8QH6PZRH8,Dubai,9953,Apartment,Residential Rent,Yearly,1 Bed,120000.0,...,124261.858238,True,low,False,not_outlier,True,high,4,4,4
1,2,http://propertyfinder.ae/to/1HMRH8078QA4JVEA65...,1HMRH8078QA4JVEA658NRM4ESW,Dubai,14794,Apartment,Residential Sale,Sale,3 Bed,3300000.0,...,72283.917753,True,low,False,not_outlier,True,high,4,4,4
2,3,http://propertyfinder.ae/to/01K96Y0WC5RMXCFN1K...,01K96Y0WC5RMXCFN1KYCTSENZV,Ajman,1703,Land,Commercial Rent,Yearly,N/A,12600000.0,...,23999.800000,True,high,False,not_outlier,True,low,6,6,6
3,4,http://propertyfinder.ae/to/01JY41ZN6BKSVN3JX3...,01JY41ZN6BKSVN3JX3X35G7XPD,Dubai,14681,Office Space,Commercial Rent,Yearly,1 Bed,110000.0,...,4807.145000,True,low,False,not_outlier,True,high,6,6,6
4,5,http://propertyfinder.ae/to/01K3389JJPGGW1N61D...,01K3389JJPGGW1N61D8W3ZSSYQ,Dubai,2685,Apartment,Residential Rent,Yearly,Studio,875222222.0,...,-0.016234,True,high,True,high,False,not_outlier,3,3,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4279,4280,http://propertyfinder.ae/to/A8TRJ9BH8CZSR57GR0...,A8TRJ9BH8CZSR57GR0E8S99WGG,Abu Dhabi,9621,Apartment,Residential Sale,Sale,Studio,890000.0,...,1.006266,False,not_outlier,False,not_outlier,True,high,3,3,3
4280,4281,http://propertyfinder.ae/to/PN40Y2KD6FEAKTEFYS...,PN40Y2KD6FEAKTEFYSKWAJVKAG,Umm Al Quwain,18095,Apartment,Residential Sale,Sale,3 Bed,1790000.0,...,-0.088316,False,not_outlier,True,low,False,not_outlier,3,3,3
4281,4282,http://propertyfinder.ae/to/8AYAC5WT6DCYFMPEW6...,8AYAC5WT6DCYFMPEW6NG9D35W4,Dubai,9081,Villa,Commercial Rent,Yearly,N/A,1500000.0,...,2.125000,False,not_outlier,False,not_outlier,True,high,3,3,3
4282,4283,http://propertyfinder.ae/to/SA76HBBNEDZ3HMXYTM...,SA76HBBNEDZ3HMXYTM0TDTRZ9C,Dubai,3666,Apartment,Residential Rent,Yearly,1 Bed,270000.0,...,1.018629,False,not_outlier,False,not_outlier,True,high,3,3,3


## Checking if price bounds can get us the pts outliers

In [6]:
inference_listings = pd.read_parquet(all_listings_path)
inference_listings

,property_listing_id,is_realistic,reason,location_id,location_lvl_0_id,location_lvl_0_name,location_lvl_1_id,location_lvl_1_name,location_lvl_2_id,location_lvl_2_name,...,price_outlier_type,price_deviation_ratio,is_sqft_outlier,sqft_outlier_type,sqft_deviation_ratio,is_any_outlier,pts_normalized_score,price_normalized_score,sqft_normalized_score,outlier_rank
0,01K99ZNNB4MN1YDMCA3SX3NBM3,None,None,15708,1,Dubai,117,City of Arabia,15708,Azizi Milan Heights,...,not_outlier,1.013131,False,not_outlier,0.993865,False,0.008350,0.004377,-0.002045,NaN
1,01KA0JV4HVMW5Q2GBR7HFR99TT,None,None,4762,3,Ras Al Khaimah,151,Al Hamra Village,1475,Royal Breeze,...,not_outlier,1.263158,False,not_outlier,1.291391,False,-0.003304,0.087719,0.097130,NaN
2,01K5XYBVY1YQ5BMWKH70EZHRQK,None,None,14132,1,Dubai,9754,Dubai Harbour,14131,Dubai Harbour Residences,...,not_outlier,0.897571,False,not_outlier,0.821038,False,0.010170,-0.051214,-0.089481,NaN
3,01K6N0JAVXZVPJ3ZR5D70AYS9V,None,None,15369,1,Dubai,49,Dubai Land,11767,California Village,...,not_outlier,1.134279,False,not_outlier,1.172563,False,-0.006168,0.044760,0.057521,NaN
4,01K62TAH7A28HRPG069APC4PPR,None,None,470,1,Dubai,470,Al Manara,None,None,...,not_outlier,0.777778,False,not_outlier,0.961538,False,-0.041667,-0.074074,-0.012821,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
350313,ZAQ9DXR9FHP9TXCANT4RKETRQ4,None,None,5269,5,Ajman,241,Al Bustan,1757,Orient Towers,...,not_outlier,0.538462,False,not_outlier,0.750000,False,-0.026316,-0.153846,-0.083333,NaN
350314,VXDCF9AP42FF3EHNSSX1E0V0BM,None,None,12521,1,Dubai,71,Jumeirah Lake Towers,12521,Orra The Embankment,...,not_outlier,1.000000,False,not_outlier,0.985333,False,0.012481,0.000000,-0.004889,NaN
350315,TM3XVVJTGXG7MJPHSV0ANHRR88,None,None,12743,6,Abu Dhabi,279,Al Reem Island,1872,City Of Lights,...,not_outlier,1.029412,False,not_outlier,1.141782,False,-0.049501,0.009804,0.047261,NaN
350316,VAEHPNENMHGTKYBQ6T75H6XRP4,None,None,10806,5,Ajman,247,Al Rawda,1771,Al Rawda 2,...,not_outlier,0.892857,False,not_outlier,1.000000,False,-0.036538,-0.021429,0.000000,NaN


In [7]:
((inference_listings['is_pts_outlier']==True) & (inference_listings['is_sqft_outlier']==False)).sum()

np.int64(720)

In [8]:
df_copy_bounds = inference_listings.copy()
natural_price_lower_bound = (1.0/df_copy_bounds['price_multiplier'])*df_copy_bounds['price_median']
natural_price_upper_bound = df_copy_bounds['price_multiplier']*df_copy_bounds['price_median']

natural_pts_lower_bound = (1.0/df_copy_bounds['pts_multiplier'])*df_copy_bounds['price_to_sqft_median']
natural_pts_upper_bound = df_copy_bounds['pts_multiplier']*df_copy_bounds['price_to_sqft_median']


In [9]:
computed_price_lower_bound = natural_pts_lower_bound*df_copy_bounds['property_sqft']
computed_price_upper_bound = natural_pts_upper_bound*df_copy_bounds['property_sqft']

In [10]:
df_copy_bounds['computed_price_outliers'] = (df_copy_bounds['price'] < computed_price_lower_bound) | (df_copy_bounds['price'] > computed_price_upper_bound)
df_copy_bounds

,property_listing_id,is_realistic,reason,location_id,location_lvl_0_id,location_lvl_0_name,location_lvl_1_id,location_lvl_1_name,location_lvl_2_id,location_lvl_2_name,...,price_deviation_ratio,is_sqft_outlier,sqft_outlier_type,sqft_deviation_ratio,is_any_outlier,pts_normalized_score,price_normalized_score,sqft_normalized_score,outlier_rank,computed_price_outliers
0,01K99ZNNB4MN1YDMCA3SX3NBM3,None,None,15708,1,Dubai,117,City of Arabia,15708,Azizi Milan Heights,...,1.013131,False,not_outlier,0.993865,False,0.008350,0.004377,-0.002045,NaN,False
1,01KA0JV4HVMW5Q2GBR7HFR99TT,None,None,4762,3,Ras Al Khaimah,151,Al Hamra Village,1475,Royal Breeze,...,1.263158,False,not_outlier,1.291391,False,-0.003304,0.087719,0.097130,NaN,False
2,01K5XYBVY1YQ5BMWKH70EZHRQK,None,None,14132,1,Dubai,9754,Dubai Harbour,14131,Dubai Harbour Residences,...,0.897571,False,not_outlier,0.821038,False,0.010170,-0.051214,-0.089481,NaN,False
3,01K6N0JAVXZVPJ3ZR5D70AYS9V,None,None,15369,1,Dubai,49,Dubai Land,11767,California Village,...,1.134279,False,not_outlier,1.172563,False,-0.006168,0.044760,0.057521,NaN,False
4,01K62TAH7A28HRPG069APC4PPR,None,None,470,1,Dubai,470,Al Manara,None,None,...,0.777778,False,not_outlier,0.961538,False,-0.041667,-0.074074,-0.012821,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
350313,ZAQ9DXR9FHP9TXCANT4RKETRQ4,None,None,5269,5,Ajman,241,Al Bustan,1757,Orient Towers,...,0.538462,False,not_outlier,0.750000,False,-0.026316,-0.153846,-0.083333,NaN,False
350314,VXDCF9AP42FF3EHNSSX1E0V0BM,None,None,12521,1,Dubai,71,Jumeirah Lake Towers,12521,Orra The Embankment,...,1.000000,False,not_outlier,0.985333,False,0.012481,0.000000,-0.004889,NaN,False
350315,TM3XVVJTGXG7MJPHSV0ANHRR88,None,None,12743,6,Abu Dhabi,279,Al Reem Island,1872,City Of Lights,...,1.029412,False,not_outlier,1.141782,False,-0.049501,0.009804,0.047261,NaN,False
350316,VAEHPNENMHGTKYBQ6T75H6XRP4,None,None,10806,5,Ajman,247,Al Rawda,1771,Al Rawda 2,...,0.892857,False,not_outlier,1.000000,False,-0.036538,-0.021429,0.000000,NaN,False


In [11]:
(df_copy_bounds['is_pts_outlier']==df_copy_bounds['computed_price_outliers'])

0         True
1         True
2         True
3         True
4         True
          ... 
350313    True
350314    True
350315    True
350316    True
350317    True
Length: 350318, dtype: bool

In [12]:
df_copy = df.copy()

In [13]:
df = df_copy.copy()
pts_df = df[df['is_pts_outlier']==True]
pts_df

,outlier_rank,url,property_listing_id,location_lvl_0_name,location_id,housing_type_name,offering_type_name,property_price_type_name,bedrooms,price,...,sqft_normalized_score,is_pts_outlier,pts_outlier_type,is_price_outlier,price_outlier_type,is_sqft_outlier,sqft_outlier_type,pts_multiplier,price_multiplier,sqft_multiplier
0,1,http://propertyfinder.ae/to/DC6YT71SJBH0WJZ1R8...,DC6YT71SJBH0WJZ1R8QH6PZRH8,Dubai,9953,Apartment,Residential Rent,Yearly,1 Bed,120000.0,...,124261.858238,True,low,False,not_outlier,True,high,4,4,4
1,2,http://propertyfinder.ae/to/1HMRH8078QA4JVEA65...,1HMRH8078QA4JVEA658NRM4ESW,Dubai,14794,Apartment,Residential Sale,Sale,3 Bed,3300000.0,...,72283.917753,True,low,False,not_outlier,True,high,4,4,4
2,3,http://propertyfinder.ae/to/01K96Y0WC5RMXCFN1K...,01K96Y0WC5RMXCFN1KYCTSENZV,Ajman,1703,Land,Commercial Rent,Yearly,N/A,12600000.0,...,23999.800000,True,high,False,not_outlier,True,low,6,6,6
3,4,http://propertyfinder.ae/to/01JY41ZN6BKSVN3JX3...,01JY41ZN6BKSVN3JX3X35G7XPD,Dubai,14681,Office Space,Commercial Rent,Yearly,1 Bed,110000.0,...,4807.145000,True,low,False,not_outlier,True,high,6,6,6
4,5,http://propertyfinder.ae/to/01K3389JJPGGW1N61D...,01K3389JJPGGW1N61D8W3ZSSYQ,Dubai,2685,Apartment,Residential Rent,Yearly,Studio,875222222.0,...,-0.016234,True,high,True,high,False,not_outlier,3,3,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2071,2072,http://propertyfinder.ae/to/DWB5PSCK9TJWY1JS4B...,DWB5PSCK9TJWY1JS4BGP2FR0QR,Sharjah,11578,Apartment,Residential Sale,Sale,2 Bed,320000.0,...,-0.015756,True,low,True,low,False,not_outlier,4,4,4
2072,2073,http://propertyfinder.ae/to/8V3GFE2DRZ7ZC36TZ6...,8V3GFE2DRZ7ZC36TZ6287XZ6B8,Ajman,1770,Villa,Residential Sale,Sale,6 Bed,4200000.0,...,-0.198600,True,high,False,not_outlier,False,not_outlier,3,3,3
2073,2074,http://propertyfinder.ae/to/HVHP8GWH1SEBC31Q38...,HVHP8GWH1SEBC31Q38VTNC2RAG,Dubai,9104,Villa,Residential Rent,Yearly,6 Bed,15000000.0,...,2.333333,True,high,True,high,True,high,4,4,4
2074,2075,http://propertyfinder.ae/to/55TKGT13V344FR8M1Y...,55TKGT13V344FR8M1YN6M6M6F0,Sharjah,13273,Apartment,Residential Sale,Sale,2 Bed,290000.0,...,0.000248,True,low,False,not_outlier,False,not_outlier,4,4,4


In [14]:
from configs.outlier_config import TOP_RESIDENTIAL_TYPES
# Get only top residential types (Apartment and Villa)
top_residential_types = TOP_RESIDENTIAL_TYPES
pts_res_df = pts_df[pts_df['housing_type_name'].isin(top_residential_types)]
pts_res_df

,outlier_rank,url,property_listing_id,location_lvl_0_name,location_id,housing_type_name,offering_type_name,property_price_type_name,bedrooms,price,...,sqft_normalized_score,is_pts_outlier,pts_outlier_type,is_price_outlier,price_outlier_type,is_sqft_outlier,sqft_outlier_type,pts_multiplier,price_multiplier,sqft_multiplier
0,1,http://propertyfinder.ae/to/DC6YT71SJBH0WJZ1R8...,DC6YT71SJBH0WJZ1R8QH6PZRH8,Dubai,9953,Apartment,Residential Rent,Yearly,1 Bed,120000.0,...,124261.858238,True,low,False,not_outlier,True,high,4,4,4
1,2,http://propertyfinder.ae/to/1HMRH8078QA4JVEA65...,1HMRH8078QA4JVEA658NRM4ESW,Dubai,14794,Apartment,Residential Sale,Sale,3 Bed,3300000.0,...,72283.917753,True,low,False,not_outlier,True,high,4,4,4
4,5,http://propertyfinder.ae/to/01K3389JJPGGW1N61D...,01K3389JJPGGW1N61D8W3ZSSYQ,Dubai,2685,Apartment,Residential Rent,Yearly,Studio,875222222.0,...,-0.016234,True,high,True,high,False,not_outlier,3,3,3
6,7,http://propertyfinder.ae/to/KFKZGRJ1YF2B4KNR72...,KFKZGRJ1YF2B4KNR72XS57K4JG,Abu Dhabi,13005,Apartment,Residential Sale,Sale,Studio,1125000.0,...,3245.216459,True,low,False,not_outlier,True,high,4,4,4
7,8,http://propertyfinder.ae/to/6SGAMTE9CMXQ3ZRKHX...,6SGAMTE9CMXQ3ZRKHXJ10E7SG4,Ajman,8612,Villa,Residential Sale,Sale,4 Bed,1600000.0,...,3333.333333,True,low,False,not_outlier,True,high,4,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2070,2071,http://propertyfinder.ae/to/208EQ1FSGQNZTWYJE2...,208EQ1FSGQNZTWYJE2AEEQ6SQC,Dubai,9525,Apartment,Residential Rent,Yearly,1 Bed,90000.0,...,-0.245343,True,high,False,not_outlier,False,not_outlier,4,4,4
2071,2072,http://propertyfinder.ae/to/DWB5PSCK9TJWY1JS4B...,DWB5PSCK9TJWY1JS4BGP2FR0QR,Sharjah,11578,Apartment,Residential Sale,Sale,2 Bed,320000.0,...,-0.015756,True,low,True,low,False,not_outlier,4,4,4
2072,2073,http://propertyfinder.ae/to/8V3GFE2DRZ7ZC36TZ6...,8V3GFE2DRZ7ZC36TZ6287XZ6B8,Ajman,1770,Villa,Residential Sale,Sale,6 Bed,4200000.0,...,-0.198600,True,high,False,not_outlier,False,not_outlier,3,3,3
2073,2074,http://propertyfinder.ae/to/HVHP8GWH1SEBC31Q38...,HVHP8GWH1SEBC31Q38VTNC2RAG,Dubai,9104,Villa,Residential Rent,Yearly,6 Bed,15000000.0,...,2.333333,True,high,True,high,True,high,4,4,4


In [15]:
pts_res_df[pts_res_df['is_sqft_outlier']==False]

,outlier_rank,url,property_listing_id,location_lvl_0_name,location_id,housing_type_name,offering_type_name,property_price_type_name,bedrooms,price,...,sqft_normalized_score,is_pts_outlier,pts_outlier_type,is_price_outlier,price_outlier_type,is_sqft_outlier,sqft_outlier_type,pts_multiplier,price_multiplier,sqft_multiplier
4,5,http://propertyfinder.ae/to/01K3389JJPGGW1N61D...,01K3389JJPGGW1N61D8W3ZSSYQ,Dubai,2685,Apartment,Residential Rent,Yearly,Studio,875222222.0,...,-0.016234,True,high,True,high,False,not_outlier,3,3,3
8,9,http://propertyfinder.ae/to/01K33V35P99FPRM5T7...,01K33V35P99FPRM5T7VD42HTGD,Dubai,2664,Apartment,Residential Rent,Yearly,Studio,755860686.0,...,-0.015901,True,high,True,high,False,not_outlier,4,4,4
45,46,http://propertyfinder.ae/to/01K3142XDVEGX77ZJQ...,01K3142XDVEGX77ZJQE2TX75HF,Dubai,2685,Apartment,Residential Rent,Yearly,Studio,87514411.0,...,-0.016234,True,high,True,high,False,not_outlier,3,3,3
51,52,http://propertyfinder.ae/to/GMESD7AR9DY797T3CV...,GMESD7AR9DY797T3CVBNYE3WHW,Sharjah,1541,Apartment,Residential Rent,Yearly,1 Bed,47500000.0,...,-0.066667,True,high,True,high,False,not_outlier,4,4,4
94,95,http://propertyfinder.ae/to/01K33YK0M63XABNPX9...,01K33YK0M63XABNPX9030JWXNT,Dubai,2664,Apartment,Residential Rent,Yearly,Studio,98500000.0,...,0.006649,True,high,True,high,False,not_outlier,4,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2069,2070,http://propertyfinder.ae/to/01JS2KEMNX646XW2ZE...,01JS2KEMNX646XW2ZE64JJG4GH,Sharjah,12571,Apartment,Residential Rent,Monthly,Studio,4500.0,...,0.833333,True,low,False,not_outlier,False,not_outlier,3,3,3
2070,2071,http://propertyfinder.ae/to/208EQ1FSGQNZTWYJE2...,208EQ1FSGQNZTWYJE2AEEQ6SQC,Dubai,9525,Apartment,Residential Rent,Yearly,1 Bed,90000.0,...,-0.245343,True,high,False,not_outlier,False,not_outlier,4,4,4
2071,2072,http://propertyfinder.ae/to/DWB5PSCK9TJWY1JS4B...,DWB5PSCK9TJWY1JS4BGP2FR0QR,Sharjah,11578,Apartment,Residential Sale,Sale,2 Bed,320000.0,...,-0.015756,True,low,True,low,False,not_outlier,4,4,4
2072,2073,http://propertyfinder.ae/to/8V3GFE2DRZ7ZC36TZ6...,8V3GFE2DRZ7ZC36TZ6287XZ6B8,Ajman,1770,Villa,Residential Sale,Sale,6 Bed,4200000.0,...,-0.198600,True,high,False,not_outlier,False,not_outlier,3,3,3


In [16]:
pts_non_res = pts_df[~pts_df['housing_type_name'].isin(top_residential_types)]
pts_non_res

,outlier_rank,url,property_listing_id,location_lvl_0_name,location_id,housing_type_name,offering_type_name,property_price_type_name,bedrooms,price,...,sqft_normalized_score,is_pts_outlier,pts_outlier_type,is_price_outlier,price_outlier_type,is_sqft_outlier,sqft_outlier_type,pts_multiplier,price_multiplier,sqft_multiplier
2,3,http://propertyfinder.ae/to/01K96Y0WC5RMXCFN1K...,01K96Y0WC5RMXCFN1KYCTSENZV,Ajman,1703,Land,Commercial Rent,Yearly,N/A,12600000.0,...,23999.800000,True,high,False,not_outlier,True,low,6,6,6
3,4,http://propertyfinder.ae/to/01JY41ZN6BKSVN3JX3...,01JY41ZN6BKSVN3JX3X35G7XPD,Dubai,14681,Office Space,Commercial Rent,Yearly,1 Bed,110000.0,...,4807.145000,True,low,False,not_outlier,True,high,6,6,6
5,6,http://propertyfinder.ae/to/0ERE8BZHVY0FHXT4V0...,0ERE8BZHVY0FHXT4V0ABKQAS1G,Umm Al Quwain,10098,Warehouse,Commercial Rent,Yearly,N/A,950000.0,...,1299.800000,True,high,False,not_outlier,True,low,6,6,6
13,14,http://propertyfinder.ae/to/DYWGQG1M19QYHYZF63...,DYWGQG1M19QYHYZF634CH5HXDG,Sharjah,1587,Warehouse,Commercial Rent,Yearly,N/A,325000.0,...,583.300000,True,high,False,not_outlier,True,low,6,6,6
15,16,http://propertyfinder.ae/to/01KC0S4BGAGGYH1WC1...,01KC0S4BGAGGYH1WC1N2D0V2Q0,Abu Dhabi,8331,Warehouse,Commercial Rent,Yearly,N/A,378750.0,...,1303.300000,True,high,False,not_outlier,True,low,6,6,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2049,2050,http://propertyfinder.ae/to/01K342RQXMEH81TZ3K...,01K342RQXMEH81TZ3KNA79YEAK,Sharjah,15031,Warehouse,Commercial Rent,Yearly,N/A,75000.0,...,0.244444,True,low,False,not_outlier,False,not_outlier,6,6,6
2055,2056,http://propertyfinder.ae/to/P799V6N9K84T5TJGK8...,P799V6N9K84T5TJGK8PBNRFJ7C,Ajman,11217,Warehouse,Commercial Sale,Sale,N/A,300000.0,...,1.736444,True,low,True,low,True,low,6,6,6
2062,2063,http://propertyfinder.ae/to/9F3JXPPDB7T9TMGB6N...,9F3JXPPDB7T9TMGB6NRDBG1228,Abu Dhabi,1991,Warehouse,Commercial Rent,Yearly,N/A,150000.0,...,1.199474,True,high,False,not_outlier,True,low,6,6,6
2065,2066,http://propertyfinder.ae/to/D762WFRZKW8XZKW3EW...,D762WFRZKW8XZKW3EWNZR6MHMW,Dubai,4410,Office Space,Commercial Rent,Yearly,N/A,65000.0,...,0.950204,True,low,False,not_outlier,False,not_outlier,6,6,6


In [17]:
pts_non_res[pts_non_res['is_sqft_outlier']==False]

,outlier_rank,url,property_listing_id,location_lvl_0_name,location_id,housing_type_name,offering_type_name,property_price_type_name,bedrooms,price,...,sqft_normalized_score,is_pts_outlier,pts_outlier_type,is_price_outlier,price_outlier_type,is_sqft_outlier,sqft_outlier_type,pts_multiplier,price_multiplier,sqft_multiplier
123,124,http://propertyfinder.ae/to/01JWTP1WBFBCRTAMN2...,01JWTP1WBFBCRTAMN2FT8GJMYP,Ajman,8612,Land,Residential Sale,Sale,N/A,670000000.0,...,0.048321,True,high,True,high,False,not_outlier,6,6,6
207,208,http://propertyfinder.ae/to/3Z58A7NTZMV313S7VF...,3Z58A7NTZMV313S7VFR9R02QBM,Ajman,5163,Whole Building,Commercial Sale,Sale,N/A,250000000.0,...,-0.163508,True,high,True,high,False,not_outlier,6,6,6
230,231,http://propertyfinder.ae/to/01KAXJBDEW3T86X10F...,01KAXJBDEW3T86X10FDJBYM2AM,Ras Al Khaimah,15568,Retail,Commercial Sale,Sale,N/A,324910474.0,...,0.047719,True,high,True,high,False,not_outlier,6,6,6
375,376,http://propertyfinder.ae/to/EF9AGWTEKXJB2HSCE2...,EF9AGWTEKXJB2HSCE2NVH880BG,Sharjah,1591,Factory,Commercial Rent,Yearly,N/A,40000000.0,...,-0.090000,True,high,True,high,False,not_outlier,6,6,6
428,429,http://propertyfinder.ae/to/SPA0XBP13MY6NVXETD...,SPA0XBP13MY6NVXETDPC0V9XJ4,Sharjah,14482,Land,Commercial Rent,Yearly,7+ Beds,50000000.0,...,0.436364,True,high,True,high,False,not_outlier,6,6,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1964,1965,http://propertyfinder.ae/to/Y50NT99GNVAAZB27Q1...,Y50NT99GNVAAZB27Q1899XE7G8,Ajman,11219,Land,Residential Sale,Sale,N/A,16000000.0,...,-0.026582,True,high,False,not_outlier,False,not_outlier,6,6,6
1968,1969,http://propertyfinder.ae/to/34WF2344Q5NRJ5JMYZ...,34WF2344Q5NRJ5JMYZ2HM95H48,Ajman,5132,Labor Camp,Commercial Rent,Yearly,N/A,40200.0,...,-0.046154,True,low,True,low,False,not_outlier,6,6,6
1990,1991,http://propertyfinder.ae/to/Y33Z12P164EQPB9SGM...,Y33Z12P164EQPB9SGM2ZDYRAJM,Dubai,2437,Office Space,Commercial Rent,Yearly,N/A,46000.0,...,0.021649,True,low,True,low,False,not_outlier,6,6,6
2049,2050,http://propertyfinder.ae/to/01K342RQXMEH81TZ3K...,01K342RQXMEH81TZ3KNA79YEAK,Sharjah,15031,Warehouse,Commercial Rent,Yearly,N/A,75000.0,...,0.244444,True,low,False,not_outlier,False,not_outlier,6,6,6


In [18]:
pts_non_res[(pts_non_res['is_sqft_outlier']==False) & (pts_non_res['housing_type_name']=='Whole Building')]


,outlier_rank,url,property_listing_id,location_lvl_0_name,location_id,housing_type_name,offering_type_name,property_price_type_name,bedrooms,price,...,sqft_normalized_score,is_pts_outlier,pts_outlier_type,is_price_outlier,price_outlier_type,is_sqft_outlier,sqft_outlier_type,pts_multiplier,price_multiplier,sqft_multiplier
207,208,http://propertyfinder.ae/to/3Z58A7NTZMV313S7VF...,3Z58A7NTZMV313S7VFR9R02QBM,Ajman,5163,Whole Building,Commercial Sale,Sale,N/A,250000000.0,...,-0.163508,True,high,True,high,False,not_outlier,6,6,6
603,604,http://propertyfinder.ae/to/A12R1YXSPRP1H5W0S8...,A12R1YXSPRP1H5W0S8G6VR0X2M,Ajman,10556,Whole Building,Commercial Sale,Sale,N/A,250000000.0,...,-0.065926,True,high,True,high,False,not_outlier,6,6,6
604,605,http://propertyfinder.ae/to/GE27DM5PNJ1QG4DWQR...,GE27DM5PNJ1QG4DWQRBZT20TTG,Ajman,10556,Whole Building,Commercial Sale,Sale,N/A,250000000.0,...,-0.065926,True,high,True,high,False,not_outlier,6,6,6
874,875,http://propertyfinder.ae/to/GBW4ZQB0G6WFMY5MHH...,GBW4ZQB0G6WFMY5MHHGFXV29HG,Ajman,10556,Whole Building,Commercial Sale,Sale,7+ Beds,666000000.0,...,0.348148,True,high,True,high,False,not_outlier,6,6,6
973,974,http://propertyfinder.ae/to/T1VKZ3PDG2CQWAJTKH...,T1VKZ3PDG2CQWAJTKHXFG9WZXG,Ajman,5115,Whole Building,Commercial Sale,Sale,N/A,300000000.0,...,0.514286,True,high,True,high,False,not_outlier,6,6,6
1823,1824,http://propertyfinder.ae/to/01K9HAR5XSMDXEPTF1...,01K9HAR5XSMDXEPTF17H34N2BK,Ajman,10937,Whole Building,Residential Sale,Sale,7+ Beds,2500000.0,...,-0.166667,True,high,False,not_outlier,False,not_outlier,6,6,6
1946,1947,http://propertyfinder.ae/to/CTS066R8EP3JMP0Q4A...,CTS066R8EP3JMP0Q4A2SS7T7XW,Ajman,5163,Whole Building,Commercial Sale,Sale,N/A,145000000.0,...,0.450598,True,high,True,high,False,not_outlier,6,6,6


selected 500-ish listings

In [19]:
if listings_version == 'pts_non_sqft_outliers':
    df = pts_df[(pts_df['is_sqft_outlier']==False)]
elif listings_version == 'pts_and_sqft_outliers':
    df = pts_df[(pts_df['is_sqft_outlier']==True)]
elif listings_version == 'pts_outliers':
    df = pts_df
df

,outlier_rank,url,property_listing_id,location_lvl_0_name,location_id,housing_type_name,offering_type_name,property_price_type_name,bedrooms,price,...,sqft_normalized_score,is_pts_outlier,pts_outlier_type,is_price_outlier,price_outlier_type,is_sqft_outlier,sqft_outlier_type,pts_multiplier,price_multiplier,sqft_multiplier
0,1,http://propertyfinder.ae/to/DC6YT71SJBH0WJZ1R8...,DC6YT71SJBH0WJZ1R8QH6PZRH8,Dubai,9953,Apartment,Residential Rent,Yearly,1 Bed,120000.0,...,124261.858238,True,low,False,not_outlier,True,high,4,4,4
1,2,http://propertyfinder.ae/to/1HMRH8078QA4JVEA65...,1HMRH8078QA4JVEA658NRM4ESW,Dubai,14794,Apartment,Residential Sale,Sale,3 Bed,3300000.0,...,72283.917753,True,low,False,not_outlier,True,high,4,4,4
2,3,http://propertyfinder.ae/to/01K96Y0WC5RMXCFN1K...,01K96Y0WC5RMXCFN1KYCTSENZV,Ajman,1703,Land,Commercial Rent,Yearly,N/A,12600000.0,...,23999.800000,True,high,False,not_outlier,True,low,6,6,6
3,4,http://propertyfinder.ae/to/01JY41ZN6BKSVN3JX3...,01JY41ZN6BKSVN3JX3X35G7XPD,Dubai,14681,Office Space,Commercial Rent,Yearly,1 Bed,110000.0,...,4807.145000,True,low,False,not_outlier,True,high,6,6,6
4,5,http://propertyfinder.ae/to/01K3389JJPGGW1N61D...,01K3389JJPGGW1N61D8W3ZSSYQ,Dubai,2685,Apartment,Residential Rent,Yearly,Studio,875222222.0,...,-0.016234,True,high,True,high,False,not_outlier,3,3,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2071,2072,http://propertyfinder.ae/to/DWB5PSCK9TJWY1JS4B...,DWB5PSCK9TJWY1JS4BGP2FR0QR,Sharjah,11578,Apartment,Residential Sale,Sale,2 Bed,320000.0,...,-0.015756,True,low,True,low,False,not_outlier,4,4,4
2072,2073,http://propertyfinder.ae/to/8V3GFE2DRZ7ZC36TZ6...,8V3GFE2DRZ7ZC36TZ6287XZ6B8,Ajman,1770,Villa,Residential Sale,Sale,6 Bed,4200000.0,...,-0.198600,True,high,False,not_outlier,False,not_outlier,3,3,3
2073,2074,http://propertyfinder.ae/to/HVHP8GWH1SEBC31Q38...,HVHP8GWH1SEBC31Q38VTNC2RAG,Dubai,9104,Villa,Residential Rent,Yearly,6 Bed,15000000.0,...,2.333333,True,high,True,high,True,high,4,4,4
2074,2075,http://propertyfinder.ae/to/55TKGT13V344FR8M1Y...,55TKGT13V344FR8M1YN6M6M6F0,Sharjah,13273,Apartment,Residential Sale,Sale,2 Bed,290000.0,...,0.000248,True,low,False,not_outlier,False,not_outlier,4,4,4


In [20]:
if check_online_status:
    # Check status for each outlier using concurrent requests
    import concurrent.futures

    print("\nChecking listing status (using parallel requests)...")


    def check_with_index(args):
        idx, url = args
        return idx, check_listing_status(url)


    # Use ThreadPoolExecutor for parallel HTTP requests
    with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
        results = list(tqdm(
            executor.map(check_with_index, enumerate(df['url'])),
            total=len(df),
            desc="Checking listings"
        ))

    # Sort by index and extract statuses
    results.sort(key=lambda x: x[0])
    statuses = [r[1] for r in results]

    df['status'] = statuses

    # Summary statistics
    print("\n" + "=" * 60)
    print("SUMMARY")
    print("=" * 60)
    print(f"\nTotal outliers checked: {len(df):,}")
    print("\nStatus breakdown:")
    print(df['status'].value_counts())
else:
    df['status'] = 'unchecked'


Checking listing status (using parallel requests)...


Checking listings: 100%|██████████| 2076/2076 [00:54<00:00, 37.95it/s] 


SUMMARY

Total outliers checked: 2,076

Status breakdown:
status
Active                                                       1931
Taken Down                                                    141
Error: HTTPSConnectionPool(host='www.propertyfinder.ae',        2
Error: HTTPSConnectionPool(host='propertyfinder.ae', port       1
Error: HTTPConnectionPool(host='propertyfinder.ae', port=       1
Name: count, dtype: int64


In [21]:
df.loc[~df['status'].isin(['Active', 'Taken Down']), 'status']='0 unchecked'
df

,outlier_rank,url,property_listing_id,location_lvl_0_name,location_id,housing_type_name,offering_type_name,property_price_type_name,bedrooms,price,...,is_pts_outlier,pts_outlier_type,is_price_outlier,price_outlier_type,is_sqft_outlier,sqft_outlier_type,pts_multiplier,price_multiplier,sqft_multiplier,status
0,1,http://propertyfinder.ae/to/DC6YT71SJBH0WJZ1R8...,DC6YT71SJBH0WJZ1R8QH6PZRH8,Dubai,9953,Apartment,Residential Rent,Yearly,1 Bed,120000.0,...,True,low,False,not_outlier,True,high,4,4,4,Active
1,2,http://propertyfinder.ae/to/1HMRH8078QA4JVEA65...,1HMRH8078QA4JVEA658NRM4ESW,Dubai,14794,Apartment,Residential Sale,Sale,3 Bed,3300000.0,...,True,low,False,not_outlier,True,high,4,4,4,Active
2,3,http://propertyfinder.ae/to/01K96Y0WC5RMXCFN1K...,01K96Y0WC5RMXCFN1KYCTSENZV,Ajman,1703,Land,Commercial Rent,Yearly,N/A,12600000.0,...,True,high,False,not_outlier,True,low,6,6,6,Active
3,4,http://propertyfinder.ae/to/01JY41ZN6BKSVN3JX3...,01JY41ZN6BKSVN3JX3X35G7XPD,Dubai,14681,Office Space,Commercial Rent,Yearly,1 Bed,110000.0,...,True,low,False,not_outlier,True,high,6,6,6,Active
4,5,http://propertyfinder.ae/to/01K3389JJPGGW1N61D...,01K3389JJPGGW1N61D8W3ZSSYQ,Dubai,2685,Apartment,Residential Rent,Yearly,Studio,875222222.0,...,True,high,True,high,False,not_outlier,3,3,3,Taken Down
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2071,2072,http://propertyfinder.ae/to/DWB5PSCK9TJWY1JS4B...,DWB5PSCK9TJWY1JS4BGP2FR0QR,Sharjah,11578,Apartment,Residential Sale,Sale,2 Bed,320000.0,...,True,low,True,low,False,not_outlier,4,4,4,Active
2072,2073,http://propertyfinder.ae/to/8V3GFE2DRZ7ZC36TZ6...,8V3GFE2DRZ7ZC36TZ6287XZ6B8,Ajman,1770,Villa,Residential Sale,Sale,6 Bed,4200000.0,...,True,high,False,not_outlier,False,not_outlier,3,3,3,Active
2073,2074,http://propertyfinder.ae/to/HVHP8GWH1SEBC31Q38...,HVHP8GWH1SEBC31Q38VTNC2RAG,Dubai,9104,Villa,Residential Rent,Yearly,6 Bed,15000000.0,...,True,high,True,high,True,high,4,4,4,Active
2074,2075,http://propertyfinder.ae/to/55TKGT13V344FR8M1Y...,55TKGT13V344FR8M1YN6M6M6F0,Sharjah,13273,Apartment,Residential Sale,Sale,2 Bed,290000.0,...,True,low,False,not_outlier,False,not_outlier,4,4,4,Active


In [22]:
df['property_listing_id']=='01KBZ380281NYMGECNX4GMHXES'

0       False
1       False
2       False
3       False
4       False
        ...  
2071    False
2072    False
2073    False
2074    False
2075    False
Name: property_listing_id, Length: 2076, dtype: bool

### Exporting listings

In [23]:
df = df.sort_values(by=['status', 'outlier_rank'], ascending=[True, True])
df['is_realistic'] = pd.NA
df['notes'] = pd.NA
first_cols = ['outlier_rank','url', 'is_realistic', 'notes', 'status']
df = df[first_cols + [c for c in df.columns if c not in first_cols]]
df

,outlier_rank,url,is_realistic,notes,status,property_listing_id,location_lvl_0_name,location_id,housing_type_name,offering_type_name,...,sqft_normalized_score,is_pts_outlier,pts_outlier_type,is_price_outlier,price_outlier_type,is_sqft_outlier,sqft_outlier_type,pts_multiplier,price_multiplier,sqft_multiplier
181,182,http://propertyfinder.ae/to/P60TET4G9AW2QCPVCF...,<NA>,<NA>,0 unchecked,P60TET4G9AW2QCPVCFCKAWAB9C,Dubai,14488,Apartment,Residential Sale,...,47.536131,True,low,False,not_outlier,True,high,3,3,3
604,605,http://propertyfinder.ae/to/GE27DM5PNJ1QG4DWQR...,<NA>,<NA>,0 unchecked,GE27DM5PNJ1QG4DWQRBZT20TTG,Ajman,10556,Whole Building,Commercial Sale,...,-0.065926,True,high,True,high,False,not_outlier,6,6,6
778,779,http://propertyfinder.ae/to/XHNSK6BZQHX3X9T20X...,<NA>,<NA>,0 unchecked,XHNSK6BZQHX3X9T20XXEP4AP54,Ajman,10959,Apartment,Residential Rent,...,0.133953,True,high,True,high,False,not_outlier,4,4,4
1840,1841,http://propertyfinder.ae/to/7X98P78K0SR9K911X7...,<NA>,<NA>,0 unchecked,7X98P78K0SR9K911X7HMAK7ZZG,Sharjah,1627,Apartment,Residential Rent,...,0.450980,True,low,False,not_outlier,False,not_outlier,4,4,4
0,1,http://propertyfinder.ae/to/DC6YT71SJBH0WJZ1R8...,<NA>,<NA>,Active,DC6YT71SJBH0WJZ1R8QH6PZRH8,Dubai,9953,Apartment,Residential Rent,...,124261.858238,True,low,False,not_outlier,True,high,4,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
909,910,http://propertyfinder.ae/to/01K3R7MFN2Y4VP1BCP...,<NA>,<NA>,Taken Down,01K3R7MFN2Y4VP1BCPW0X523FV,Dubai,2790,Apartment,Residential Sale,...,3.266234,True,high,False,not_outlier,True,low,4,4,4
920,921,http://propertyfinder.ae/to/01K8Q8QHJW1B4M4DZQ...,<NA>,<NA>,Taken Down,01K8Q8QHJW1B4M4DZQK5ZT23S8,Dubai,2672,Apartment,Residential Sale,...,0.071853,True,low,True,low,False,not_outlier,4,4,4
924,925,http://propertyfinder.ae/to/01K3R7FNJ76YKHMAKA...,<NA>,<NA>,Taken Down,01K3R7FNJ76YKHMAKA22BVMG21,Dubai,11726,Apartment,Residential Rent,...,2.567100,True,high,False,not_outlier,True,low,4,4,4
925,926,http://propertyfinder.ae/to/01K3R7H33ECCGQ41RS...,<NA>,<NA>,Taken Down,01K3R7H33ECCGQ41RS67MCG3TE,Dubai,13587,Apartment,Residential Rent,...,5.463470,True,high,False,not_outlier,True,low,4,4,4


Selecting only 500

In [24]:
status_checked_df = df.copy()
# df = df.head(500)
df

,outlier_rank,url,is_realistic,notes,status,property_listing_id,location_lvl_0_name,location_id,housing_type_name,offering_type_name,...,sqft_normalized_score,is_pts_outlier,pts_outlier_type,is_price_outlier,price_outlier_type,is_sqft_outlier,sqft_outlier_type,pts_multiplier,price_multiplier,sqft_multiplier
181,182,http://propertyfinder.ae/to/P60TET4G9AW2QCPVCF...,<NA>,<NA>,0 unchecked,P60TET4G9AW2QCPVCFCKAWAB9C,Dubai,14488,Apartment,Residential Sale,...,47.536131,True,low,False,not_outlier,True,high,3,3,3
604,605,http://propertyfinder.ae/to/GE27DM5PNJ1QG4DWQR...,<NA>,<NA>,0 unchecked,GE27DM5PNJ1QG4DWQRBZT20TTG,Ajman,10556,Whole Building,Commercial Sale,...,-0.065926,True,high,True,high,False,not_outlier,6,6,6
778,779,http://propertyfinder.ae/to/XHNSK6BZQHX3X9T20X...,<NA>,<NA>,0 unchecked,XHNSK6BZQHX3X9T20XXEP4AP54,Ajman,10959,Apartment,Residential Rent,...,0.133953,True,high,True,high,False,not_outlier,4,4,4
1840,1841,http://propertyfinder.ae/to/7X98P78K0SR9K911X7...,<NA>,<NA>,0 unchecked,7X98P78K0SR9K911X7HMAK7ZZG,Sharjah,1627,Apartment,Residential Rent,...,0.450980,True,low,False,not_outlier,False,not_outlier,4,4,4
0,1,http://propertyfinder.ae/to/DC6YT71SJBH0WJZ1R8...,<NA>,<NA>,Active,DC6YT71SJBH0WJZ1R8QH6PZRH8,Dubai,9953,Apartment,Residential Rent,...,124261.858238,True,low,False,not_outlier,True,high,4,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
909,910,http://propertyfinder.ae/to/01K3R7MFN2Y4VP1BCP...,<NA>,<NA>,Taken Down,01K3R7MFN2Y4VP1BCPW0X523FV,Dubai,2790,Apartment,Residential Sale,...,3.266234,True,high,False,not_outlier,True,low,4,4,4
920,921,http://propertyfinder.ae/to/01K8Q8QHJW1B4M4DZQ...,<NA>,<NA>,Taken Down,01K8Q8QHJW1B4M4DZQK5ZT23S8,Dubai,2672,Apartment,Residential Sale,...,0.071853,True,low,True,low,False,not_outlier,4,4,4
924,925,http://propertyfinder.ae/to/01K3R7FNJ76YKHMAKA...,<NA>,<NA>,Taken Down,01K3R7FNJ76YKHMAKA22BVMG21,Dubai,11726,Apartment,Residential Rent,...,2.567100,True,high,False,not_outlier,True,low,4,4,4
925,926,http://propertyfinder.ae/to/01K3R7H33ECCGQ41RS...,<NA>,<NA>,Taken Down,01K3R7H33ECCGQ41RS67MCG3TE,Dubai,13587,Apartment,Residential Rent,...,5.463470,True,high,False,not_outlier,True,low,4,4,4


In [25]:
# # Save results
# output_path = f'../datasets/{version}/{listings_version}_unrealistic_listings_{version}_status_checked.csv'
# df.to_csv(output_path, index=False)
# print(f"Results saved to: {output_path}")

### Exporting listings per emirate

Some stats

In [26]:
df[(df['status'] == 'Taken Down') & (df['is_pts_outlier'] == True)]['location_lvl_0_name'].value_counts()

location_lvl_0_name
Dubai      134
Ajman        5
Sharjah      2
Name: count, dtype: int64

In [27]:
df[(df['status'] == 'Taken Down')]['location_lvl_0_name'].value_counts()

location_lvl_0_name
Dubai      134
Ajman        5
Sharjah      2
Name: count, dtype: int64

In [28]:
df[(df['status'] == 'Taken Down')]['is_price_outlier'].value_counts()

is_price_outlier
False    76
True     65
Name: count, dtype: int64

In [29]:
df[(df['status'] == 'Taken Down')]['is_sqft_outlier'].value_counts()

is_sqft_outlier
True     80
False    61
Name: count, dtype: int64

In [30]:
df[(df['status'] == 'Taken Down')]['is_pts_outlier'].value_counts()

is_pts_outlier
True    141
Name: count, dtype: int64

290 listings taken down out of 331 were pts outliers

In [31]:
(df['is_pts_outlier'] == True).sum(), ((df['is_pts_outlier'] == True) & (df['status'] == 'Taken Down')).sum()

(np.int64(2076), np.int64(141))

In [32]:
((df['is_pts_outlier'] == True) & (df['is_sqft_outlier'] == False)).sum()

np.int64(720)

In [33]:
((df['is_pts_outlier'] == True) & (df['is_sqft_outlier'] == False) & (df['status'] == 'Active')).sum()

np.int64(656)

Exporting pts outliers

In [34]:
pts_outlier_listings = df[(df['is_pts_outlier'] == True)]
pts_outlier_listings

,outlier_rank,url,is_realistic,notes,status,property_listing_id,location_lvl_0_name,location_id,housing_type_name,offering_type_name,...,sqft_normalized_score,is_pts_outlier,pts_outlier_type,is_price_outlier,price_outlier_type,is_sqft_outlier,sqft_outlier_type,pts_multiplier,price_multiplier,sqft_multiplier
181,182,http://propertyfinder.ae/to/P60TET4G9AW2QCPVCF...,<NA>,<NA>,0 unchecked,P60TET4G9AW2QCPVCFCKAWAB9C,Dubai,14488,Apartment,Residential Sale,...,47.536131,True,low,False,not_outlier,True,high,3,3,3
604,605,http://propertyfinder.ae/to/GE27DM5PNJ1QG4DWQR...,<NA>,<NA>,0 unchecked,GE27DM5PNJ1QG4DWQRBZT20TTG,Ajman,10556,Whole Building,Commercial Sale,...,-0.065926,True,high,True,high,False,not_outlier,6,6,6
778,779,http://propertyfinder.ae/to/XHNSK6BZQHX3X9T20X...,<NA>,<NA>,0 unchecked,XHNSK6BZQHX3X9T20XXEP4AP54,Ajman,10959,Apartment,Residential Rent,...,0.133953,True,high,True,high,False,not_outlier,4,4,4
1840,1841,http://propertyfinder.ae/to/7X98P78K0SR9K911X7...,<NA>,<NA>,0 unchecked,7X98P78K0SR9K911X7HMAK7ZZG,Sharjah,1627,Apartment,Residential Rent,...,0.450980,True,low,False,not_outlier,False,not_outlier,4,4,4
0,1,http://propertyfinder.ae/to/DC6YT71SJBH0WJZ1R8...,<NA>,<NA>,Active,DC6YT71SJBH0WJZ1R8QH6PZRH8,Dubai,9953,Apartment,Residential Rent,...,124261.858238,True,low,False,not_outlier,True,high,4,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
909,910,http://propertyfinder.ae/to/01K3R7MFN2Y4VP1BCP...,<NA>,<NA>,Taken Down,01K3R7MFN2Y4VP1BCPW0X523FV,Dubai,2790,Apartment,Residential Sale,...,3.266234,True,high,False,not_outlier,True,low,4,4,4
920,921,http://propertyfinder.ae/to/01K8Q8QHJW1B4M4DZQ...,<NA>,<NA>,Taken Down,01K8Q8QHJW1B4M4DZQK5ZT23S8,Dubai,2672,Apartment,Residential Sale,...,0.071853,True,low,True,low,False,not_outlier,4,4,4
924,925,http://propertyfinder.ae/to/01K3R7FNJ76YKHMAKA...,<NA>,<NA>,Taken Down,01K3R7FNJ76YKHMAKA22BVMG21,Dubai,11726,Apartment,Residential Rent,...,2.567100,True,high,False,not_outlier,True,low,4,4,4
925,926,http://propertyfinder.ae/to/01K3R7H33ECCGQ41RS...,<NA>,<NA>,Taken Down,01K3R7H33ECCGQ41RS67MCG3TE,Dubai,13587,Apartment,Residential Rent,...,5.463470,True,high,False,not_outlier,True,low,4,4,4


In [35]:
outside_dubai_count = ((df['location_lvl_0_name'] != 'Dubai')).sum()
uae_count = len(df)
print(f'outside_dubai_count: {outside_dubai_count}')
print(f'Dubai: {uae_count-outside_dubai_count}')
print(f'UEA: {uae_count}')

outside_dubai_count: 1061
Dubai: 1015
UEA: 2076


In [36]:
((df['status'] == 'Taken Down') & (df['location_lvl_0_name'] != 'Dubai')).sum()

np.int64(7)

In [37]:
df[df['property_listing_id']=='J63EQ1DQNY83GV8K09RZSJRBWG']

,outlier_rank,url,is_realistic,notes,status,property_listing_id,location_lvl_0_name,location_id,housing_type_name,offering_type_name,...,sqft_normalized_score,is_pts_outlier,pts_outlier_type,is_price_outlier,price_outlier_type,is_sqft_outlier,sqft_outlier_type,pts_multiplier,price_multiplier,sqft_multiplier


### Get info of a certain segment

In [38]:
all_listings = pd.read_parquet(all_listings_path)
all_listings

,property_listing_id,is_realistic,reason,location_id,location_lvl_0_id,location_lvl_0_name,location_lvl_1_id,location_lvl_1_name,location_lvl_2_id,location_lvl_2_name,...,price_outlier_type,price_deviation_ratio,is_sqft_outlier,sqft_outlier_type,sqft_deviation_ratio,is_any_outlier,pts_normalized_score,price_normalized_score,sqft_normalized_score,outlier_rank
0,01K99ZNNB4MN1YDMCA3SX3NBM3,None,None,15708,1,Dubai,117,City of Arabia,15708,Azizi Milan Heights,...,not_outlier,1.013131,False,not_outlier,0.993865,False,0.008350,0.004377,-0.002045,NaN
1,01KA0JV4HVMW5Q2GBR7HFR99TT,None,None,4762,3,Ras Al Khaimah,151,Al Hamra Village,1475,Royal Breeze,...,not_outlier,1.263158,False,not_outlier,1.291391,False,-0.003304,0.087719,0.097130,NaN
2,01K5XYBVY1YQ5BMWKH70EZHRQK,None,None,14132,1,Dubai,9754,Dubai Harbour,14131,Dubai Harbour Residences,...,not_outlier,0.897571,False,not_outlier,0.821038,False,0.010170,-0.051214,-0.089481,NaN
3,01K6N0JAVXZVPJ3ZR5D70AYS9V,None,None,15369,1,Dubai,49,Dubai Land,11767,California Village,...,not_outlier,1.134279,False,not_outlier,1.172563,False,-0.006168,0.044760,0.057521,NaN
4,01K62TAH7A28HRPG069APC4PPR,None,None,470,1,Dubai,470,Al Manara,None,None,...,not_outlier,0.777778,False,not_outlier,0.961538,False,-0.041667,-0.074074,-0.012821,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
350313,ZAQ9DXR9FHP9TXCANT4RKETRQ4,None,None,5269,5,Ajman,241,Al Bustan,1757,Orient Towers,...,not_outlier,0.538462,False,not_outlier,0.750000,False,-0.026316,-0.153846,-0.083333,NaN
350314,VXDCF9AP42FF3EHNSSX1E0V0BM,None,None,12521,1,Dubai,71,Jumeirah Lake Towers,12521,Orra The Embankment,...,not_outlier,1.000000,False,not_outlier,0.985333,False,0.012481,0.000000,-0.004889,NaN
350315,TM3XVVJTGXG7MJPHSV0ANHRR88,None,None,12743,6,Abu Dhabi,279,Al Reem Island,1872,City Of Lights,...,not_outlier,1.029412,False,not_outlier,1.141782,False,-0.049501,0.009804,0.047261,NaN
350316,VAEHPNENMHGTKYBQ6T75H6XRP4,None,None,10806,5,Ajman,247,Al Rawda,1771,Al Rawda 2,...,not_outlier,0.892857,False,not_outlier,1.000000,False,-0.036538,-0.021429,0.000000,NaN


In [39]:
all_listings[all_listings['segment_key'] == '9573|Apartment|Residential Rent|Yearly']

,property_listing_id,is_realistic,reason,location_id,location_lvl_0_id,location_lvl_0_name,location_lvl_1_id,location_lvl_1_name,location_lvl_2_id,location_lvl_2_name,...,price_outlier_type,price_deviation_ratio,is_sqft_outlier,sqft_outlier_type,sqft_deviation_ratio,is_any_outlier,pts_normalized_score,price_normalized_score,sqft_normalized_score,outlier_rank
7191,01K60G1ZXND7TNM7R6W5SMSX6Z,None,None,9573,1,Dubai,71,Jumeirah Lake Towers,9573,The Residences JLT,...,not_outlier,2.673797,False,not_outlier,2.440415,False,0.016171,0.557932,0.480138,NaN
222272,CHY3ME34ZP1G7WRNSDRRCCXMYW,None,None,9573,1,Dubai,71,Jumeirah Lake Towers,9573,The Residences JLT,...,not_outlier,2.139037,False,not_outlier,1.660992,False,0.077475,0.379679,0.220331,NaN


In [40]:
online_counts = (all_listings['end_year'] == 9999).sum()
online_counts

np.int64(350318)

In [41]:
(uae_count - outside_dubai_count) / online_counts

np.float64(0.0028973675346399557)

In [42]:
outside_dubai_count / online_counts, uae_count / online_counts

(np.float64(0.0030286768022196975), np.float64(0.005926044336859653))

In [43]:
dubai_df = df[df['location_lvl_0_name'] == 'Dubai']
non_dubai_df = df[df['location_lvl_0_name'] != 'Dubai']

dubai_flagged = len(dubai_df)
dubai_taken_down = (dubai_df['status'] == 'Taken Down').sum()
non_dubai_flagged = len(non_dubai_df)
non_dubai_taken_down = (non_dubai_df['status'] == 'Taken Down').sum()

In [44]:
import os

emirates = df['location_lvl_0_name'].unique()

# Create output directory if it doesn't exist
output_dir = f'../daily_flagged_listings/detailed_{version[:-3]}_{listings_version}{feedback}'
os.makedirs(output_dir, exist_ok=True)

# Export all pts outlier listings
df.to_csv(f'{output_dir}/{listings_version}_unrealistic_listings_{version[:-3]}{feedback}_UAE.csv', index=False)
print(f"Exported all pts outliers to: {output_dir}/{listings_version}_unrealistic_listings_{version[:-3]}{feedback}_UAE.csv")

# Create summary dataframe
summary_data = []
for emirate in emirates:
    emirate_df = df[df['location_lvl_0_name'] == emirate]
    emirate_filename = emirate.lower().replace(' ', '_')
    emirate_df.to_csv(f'{output_dir}/{listings_version}_unrealistic_listings_{version[:-3]}{feedback}_{emirate_filename}.csv', index=False)
    summary_data.append({
        'Emirate': emirate,
        'Flagged': len(emirate_df),
        'Taken Down': (emirate_df['status'] == 'Taken Down').sum()
    })

# Add overall summary rows
summary_data.append({'Emirate': '---', 'Flagged': '---', 'Taken Down': '---'})
# summary_data.append({'Emirate': 'Dubai', 'Flagged': dubai_flagged, 'Taken Down': dubai_taken_down})
summary_data.append({'Emirate': 'Non-Dubai', 'Flagged': non_dubai_flagged, 'Taken Down': non_dubai_taken_down})
summary_data.append({'Emirate': 'Total UAE', 'Flagged': uae_count,
                     'Taken Down': (df['status'] == 'Taken Down').sum()})

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(f'{output_dir}/overall_stats_{listings_version}_unrealistic_listings_{version[:-3]}{feedback}.csv', index=False)
print(f"Saved summary to: {output_dir}/overall_stats_{listings_version}_unrealistic_listings_{version[:-3]}{feedback}.csv")
summary_df

Exported all pts outliers to: ../daily_flagged_listings/detailed_april_17_pts_outliers/pts_outliers_unrealistic_listings_april_17_UAE.csv
Saved summary to: ../daily_flagged_listings/detailed_april_17_pts_outliers/overall_stats_pts_outliers_unrealistic_listings_april_17.csv


,Emirate,Flagged,Taken Down
0,Dubai,1015,134
1,Ajman,479,5
2,Sharjah,406,2
3,Umm Al Quwain,21,0
4,Abu Dhabi,66,0
5,Ras Al Khaimah,87,0
6,Fujairah,2,0
7,---,---,---
8,Non-Dubai,1061,7
9,Total UAE,2076,141


In [45]:
import os

short_df = df[['url', 'property_listing_id','location_lvl_0_name', 'price_outlier_type', 'sqft_outlier_type', 'price', 'property_sqft', 'status']].copy()

emirates = short_df['location_lvl_0_name'].unique()

# Create output directory if it doesn't exist
output_dir = f'../daily_flagged_listings/{version[:-3]}_{listings_version}{feedback}'
os.makedirs(output_dir, exist_ok=True)

# Export all pts outlier listings
short_df.to_csv(f'{output_dir}/{listings_version}_unrealistic_listings_{version[:-3]}{feedback}_UAE.csv', index=False)
print(f"Exported all pts outliers to: {output_dir}/{listings_version}_unrealistic_listings_{version[:-3]}{feedback}_UAE.csv")

# Create summary dataframe
summary_data = []
for emirate in emirates:
    emirate_short_df = short_df[short_df['location_lvl_0_name'] == emirate]
    emirate_filename = emirate.lower().replace(' ', '_')
    emirate_short_df.to_csv(f'{output_dir}/{listings_version}_unrealistic_listings_{version[:-3]}{feedback}_{emirate_filename}.csv', index=False)
    summary_data.append({
        'Emirate': emirate,
        'Flagged': len(emirate_short_df),
        'Taken Down': (emirate_short_df['status'] == 'Taken Down').sum()
    })

# Add overall summary rows
summary_data.append({'Emirate': '---', 'Flagged': '---', 'Taken Down': '---'})
# summary_data.append({'Emirate': 'Dubai', 'Flagged': dubai_flagged, 'Taken Down': dubai_taken_down})
summary_data.append({'Emirate': 'Non-Dubai', 'Flagged': non_dubai_flagged, 'Taken Down': non_dubai_taken_down})
summary_data.append({'Emirate': 'Total UAE', 'Flagged': uae_count,
                     'Taken Down': (short_df['status'] == 'Taken Down').sum()})

summary_short_df = pd.DataFrame(summary_data)
summary_short_df.to_csv(f'{output_dir}/overall_stats_{listings_version}_unrealistic_listings_{version[:-3]}{feedback}.csv', index=False)
print(f"Saved summary to: {output_dir}/overall_stats_{listings_version}_unrealistic_listings_{version[:-3]}{feedback}.csv")
summary_short_df

Exported all pts outliers to: ../daily_flagged_listings/april_17_pts_outliers/pts_outliers_unrealistic_listings_april_17_UAE.csv
Saved summary to: ../daily_flagged_listings/april_17_pts_outliers/overall_stats_pts_outliers_unrealistic_listings_april_17.csv


,Emirate,Flagged,Taken Down
0,Dubai,1015,134
1,Ajman,479,5
2,Sharjah,406,2
3,Umm Al Quwain,21,0
4,Abu Dhabi,66,0
5,Ras Al Khaimah,87,0
6,Fujairah,2,0
7,---,---,---
8,Non-Dubai,1061,7
9,Total UAE,2076,141


In [46]:
newly_flagged_listings = df[((df['is_price_outlier']==True) & (df['is_pts_outlier']==False)) | ((df['is_sqft_outlier']==True) & (df['is_pts_outlier']==False))]
newly_flagged_listings

,outlier_rank,url,is_realistic,notes,status,property_listing_id,location_lvl_0_name,location_id,housing_type_name,offering_type_name,...,sqft_normalized_score,is_pts_outlier,pts_outlier_type,is_price_outlier,price_outlier_type,is_sqft_outlier,sqft_outlier_type,pts_multiplier,price_multiplier,sqft_multiplier


In [47]:
((df['is_price_outlier']==True) & (df['is_pts_outlier']==False) & (df['is_sqft_outlier']==False) ).sum()

np.int64(0)

In [48]:
((df['is_sqft_outlier']==True) & (df['is_pts_outlier']==False) & (df['is_price_outlier']==False) ).sum()

np.int64(0)

In [49]:
((df['is_price_outlier']==True) | (df['is_sqft_outlier']==True) ).sum()

np.int64(1835)

In [50]:
((df['is_sqft_outlier']==True)).sum()

np.int64(1356)

## Analysing a certain emirate

In [51]:
temp_emirate = "Abu Dhabi"
temp_emirate_stats = df[((df['is_pts_outlier']==True) & (df['location_lvl_0_name']==temp_emirate))]
temp_emirate_stats

,outlier_rank,url,is_realistic,notes,status,property_listing_id,location_lvl_0_name,location_id,housing_type_name,offering_type_name,...,sqft_normalized_score,is_pts_outlier,pts_outlier_type,is_price_outlier,price_outlier_type,is_sqft_outlier,sqft_outlier_type,pts_multiplier,price_multiplier,sqft_multiplier
6,7,http://propertyfinder.ae/to/KFKZGRJ1YF2B4KNR72...,<NA>,<NA>,Active,KFKZGRJ1YF2B4KNR72XS57K4JG,Abu Dhabi,13005,Apartment,Residential Sale,...,3245.216459,True,low,False,not_outlier,True,high,4,4,4
15,16,http://propertyfinder.ae/to/01KC0S4BGAGGYH1WC1...,<NA>,<NA>,Active,01KC0S4BGAGGYH1WC1N2D0V2Q0,Abu Dhabi,8331,Warehouse,Commercial Rent,...,1303.300000,True,high,False,not_outlier,True,low,6,6,6
37,38,http://propertyfinder.ae/to/RP5Y5RWFBNT0Q36BNP...,<NA>,<NA>,Active,RP5Y5RWFBNT0Q36BNP6XWX6ND0,Abu Dhabi,12776,Villa,Residential Sale,...,828.000000,True,high,False,not_outlier,True,low,3,3,3
87,88,http://propertyfinder.ae/to/1EB1XHET60RMEXX5S8...,<NA>,<NA>,Active,1EB1XHET60RMEXX5S898E7MWFG,Abu Dhabi,17888,Apartment,Residential Sale,...,298.487179,True,low,False,not_outlier,True,high,4,4,4
146,147,http://propertyfinder.ae/to/556XB8DGGYH2NH01PM...,<NA>,<NA>,Active,556XB8DGGYH2NH01PMV2EBHM00,Abu Dhabi,9836,Office Space,Commercial Rent,...,7.125818,True,low,False,not_outlier,True,high,6,6,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1923,1924,http://propertyfinder.ae/to/PNKBAXJP4R8GZZREQ8...,<NA>,<NA>,Active,PNKBAXJP4R8GZZREQ8C7ZDWKFG,Abu Dhabi,12776,Villa,Residential Sale,...,1.883030,True,high,False,not_outlier,True,low,3,3,3
1978,1979,http://propertyfinder.ae/to/AS2Q1PZGS7AFZVN45Q...,<NA>,<NA>,Active,AS2Q1PZGS7AFZVN45Q6D4NPFS0,Abu Dhabi,18186,Villa,Residential Sale,...,1.999064,True,high,False,not_outlier,True,low,4,4,4
1985,1986,http://propertyfinder.ae/to/01JZW5APS43CNJXTWZ...,<NA>,<NA>,Active,01JZW5APS43CNJXTWZ2NQSQ1ZQ,Abu Dhabi,12310,Villa,Residential Rent,...,1.333222,True,low,False,not_outlier,True,high,3,3,3
2027,2028,http://propertyfinder.ae/to/15GX3GS3XYRKSR7FM3...,<NA>,<NA>,Active,15GX3GS3XYRKSR7FM30M5RQ4M8,Abu Dhabi,305,Villa,Residential Rent,...,1.429343,True,low,False,not_outlier,True,high,3,3,3


In [52]:
temp_emirate_stats['sqft_outlier_type'].value_counts()

sqft_outlier_type
high           29
low            23
not_outlier    14
Name: count, dtype: int64

In [53]:
temp_emirate_stats['price_outlier_type'].value_counts()

price_outlier_type
not_outlier    59
high            7
Name: count, dtype: int64

In [54]:
temp_emirate_stats[((temp_emirate_stats['is_price_outlier']==False) & (temp_emirate_stats['is_sqft_outlier']==False))]

,outlier_rank,url,is_realistic,notes,status,property_listing_id,location_lvl_0_name,location_id,housing_type_name,offering_type_name,...,sqft_normalized_score,is_pts_outlier,pts_outlier_type,is_price_outlier,price_outlier_type,is_sqft_outlier,sqft_outlier_type,pts_multiplier,price_multiplier,sqft_multiplier
496,497,http://propertyfinder.ae/to/8HX85RXHM9EYJN4Y6F...,<NA>,<NA>,Active,8HX85RXHM9EYJN4Y6FXVYH0WXR,Abu Dhabi,9836,Office Space,Commercial Rent,...,0.189036,True,low,False,not_outlier,False,not_outlier,6,6,6
497,498,http://propertyfinder.ae/to/CSY9TZY23F2CVR7NH9...,<NA>,<NA>,Active,CSY9TZY23F2CVR7NH9C5VX2YHR,Abu Dhabi,9836,Office Space,Commercial Rent,...,0.189036,True,low,False,not_outlier,False,not_outlier,6,6,6
498,499,http://propertyfinder.ae/to/RVF826DMRESRC5WBNT...,<NA>,<NA>,Active,RVF826DMRESRC5WBNTAYJCFQXG,Abu Dhabi,9836,Office Space,Commercial Rent,...,0.189036,True,low,False,not_outlier,False,not_outlier,6,6,6
499,500,http://propertyfinder.ae/to/AEH6F4WWNHJC2VMAQ8...,<NA>,<NA>,Active,AEH6F4WWNHJC2VMAQ8BJ49KT38,Abu Dhabi,9836,Office Space,Commercial Rent,...,0.189036,True,low,False,not_outlier,False,not_outlier,6,6,6
500,501,http://propertyfinder.ae/to/2CEHWG5ASG4XFGF939...,<NA>,<NA>,Active,2CEHWG5ASG4XFGF939ZDB4NPB8,Abu Dhabi,9836,Office Space,Commercial Rent,...,0.252626,True,low,False,not_outlier,False,not_outlier,6,6,6
501,502,http://propertyfinder.ae/to/XR93MZ0J22SF5F3ZCT...,<NA>,<NA>,Active,XR93MZ0J22SF5F3ZCTXN0ACWN0,Abu Dhabi,9836,Office Space,Commercial Rent,...,0.309745,True,low,False,not_outlier,False,not_outlier,6,6,6
1842,1843,http://propertyfinder.ae/to/J2B8ZTNY09JPZGMXQT...,<NA>,<NA>,Active,J2B8ZTNY09JPZGMXQTB901F9XW,Abu Dhabi,14215,Villa,Residential Sale,...,-0.312613,True,high,False,not_outlier,False,not_outlier,3,3,3


In [55]:
temp_display = temp_emirate_stats[['url', 'property_listing_id', 'price_outlier_type', 'sqft_outlier_type', 'price', 'property_sqft']].copy()
# Replace not_outlier with 'high/low' when both types are not_outlier
both_not_outlier = (temp_display['price_outlier_type'] == 'not_outlier') & (temp_display['sqft_outlier_type'] == 'not_outlier')
temp_display.loc[both_not_outlier, 'sqft_outlier_type'] = 'high/low'
# Replace remaining not_outlier with 'No'
temp_display['price_outlier_type'] = temp_display['price_outlier_type'].replace('not_outlier', 'No')
temp_display['sqft_outlier_type'] = temp_display['sqft_outlier_type'].replace('not_outlier', 'No')
temp_display.reset_index().drop(columns='index').to_csv(f'{output_dir}/AD_flagged_listings_{listings_version}{feedback}_indexed.csv')
temp_display

,url,property_listing_id,price_outlier_type,sqft_outlier_type,price,property_sqft
6,http://propertyfinder.ae/to/KFKZGRJ1YF2B4KNR72...,KFKZGRJ1YF2B4KNR72XS57K4JG,No,high,1125000.0,4693065.0
15,http://propertyfinder.ae/to/01KC0S4BGAGGYH1WC1...,01KC0S4BGAGGYH1WC1N2D0V2Q0,No,low,378750.0,1.0
37,http://propertyfinder.ae/to/RP5Y5RWFBNT0Q36BNP...,RP5Y5RWFBNT0Q36BNP6XWX6ND0,No,low,16000000.0,3.0
87,http://propertyfinder.ae/to/1EB1XHET60RMEXX5S8...,1EB1XHET60RMEXX5S898E7MWFG,No,high,3150000.0,1212016.0
146,http://propertyfinder.ae/to/556XB8DGGYH2NH01PM...,556XB8DGGYH2NH01PMV2EBHM00,No,high,274176.0,2108004.0
...,...,...,...,...,...,...
1923,http://propertyfinder.ae/to/PNKBAXJP4R8GZZREQ8...,PNKBAXJP4R8GZZREQ8C7ZDWKFG,No,low,15000000.0,1043.0
1978,http://propertyfinder.ae/to/AS2Q1PZGS7AFZVN45Q...,AS2Q1PZGS7AFZVN45Q6D4NPFS0,No,low,6000000.0,2492.0
1985,http://propertyfinder.ae/to/01JZW5APS43CNJXTWZ...,01JZW5APS43CNJXTWZ2NQSQ1ZQ,No,high,250000.0,16499.0
2027,http://propertyfinder.ae/to/15GX3GS3XYRKSR7FM3...,15GX3GS3XYRKSR7FM30M5RQ4M8,No,high,230000.0,22500.0


# Further analysis on compliance and short term listings

In [57]:
df.columns

Index(['outlier_rank', 'url', 'is_realistic', 'notes', 'status',
       'property_listing_id', 'location_lvl_0_name', 'location_id',
       'housing_type_name', 'offering_type_name', 'property_price_type_name',
       'bedrooms', 'price', 'property_sqft', 'price_to_sqft', 'segment_key',
       'segment_count', 'relaxation_level', 'price_median',
       'property_sqft_median', 'price_to_sqft_median', 'pts_deviation_ratio',
       'price_deviation_ratio', 'sqft_deviation_ratio', 'pts_normalized_score',
       'price_normalized_score', 'sqft_normalized_score', 'is_pts_outlier',
       'pts_outlier_type', 'is_price_outlier', 'price_outlier_type',
       'is_sqft_outlier', 'sqft_outlier_type', 'pts_multiplier',
       'price_multiplier', 'sqft_multiplier'],
      dtype='object')

In [58]:
# listing_client_account = df[['property_listing_id', 'client_id', 'salesforce_account_no', 'property_permit_number', 'listing_short_term_flag', 'complaince_type']]
# listing_client_account

In [59]:
# listing_client_account.to_csv(f'{output_dir}/listings_to_client_map.csv', index=False)
# listing_client_account = pd.read_csv(f'{output_dir}/listings_to_client_map.csv')
# listing_client_account

## Exploring Non-Residential housing types

In [67]:
df_copy[~df_copy['housing_type_name'].isin(top_residential_types)]

,outlier_rank,url,property_listing_id,location_lvl_0_name,location_id,housing_type_name,offering_type_name,property_price_type_name,bedrooms,price,...,sqft_normalized_score,is_pts_outlier,pts_outlier_type,is_price_outlier,price_outlier_type,is_sqft_outlier,sqft_outlier_type,pts_multiplier,price_multiplier,sqft_multiplier
2,3,http://propertyfinder.ae/to/01K96Y0WC5RMXCFN1K...,01K96Y0WC5RMXCFN1KYCTSENZV,Ajman,1703,Land,Commercial Rent,Yearly,N/A,12600000.0,...,23999.800000,True,high,False,not_outlier,True,low,6,6,6
3,4,http://propertyfinder.ae/to/01JY41ZN6BKSVN3JX3...,01JY41ZN6BKSVN3JX3X35G7XPD,Dubai,14681,Office Space,Commercial Rent,Yearly,1 Bed,110000.0,...,4807.145000,True,low,False,not_outlier,True,high,6,6,6
5,6,http://propertyfinder.ae/to/0ERE8BZHVY0FHXT4V0...,0ERE8BZHVY0FHXT4V0ABKQAS1G,Umm Al Quwain,10098,Warehouse,Commercial Rent,Yearly,N/A,950000.0,...,1299.800000,True,high,False,not_outlier,True,low,6,6,6
13,14,http://propertyfinder.ae/to/DYWGQG1M19QYHYZF63...,DYWGQG1M19QYHYZF634CH5HXDG,Sharjah,1587,Warehouse,Commercial Rent,Yearly,N/A,325000.0,...,583.300000,True,high,False,not_outlier,True,low,6,6,6
15,16,http://propertyfinder.ae/to/01KC0S4BGAGGYH1WC1...,01KC0S4BGAGGYH1WC1N2D0V2Q0,Abu Dhabi,8331,Warehouse,Commercial Rent,Yearly,N/A,378750.0,...,1303.300000,True,high,False,not_outlier,True,low,6,6,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4134,4135,http://propertyfinder.ae/to/9Z16K5NNAB8ZD4ZSWX...,9Z16K5NNAB8ZD4ZSWXBCMHGGT8,Dubai,4409,Office Space,Commercial Rent,Yearly,N/A,60000.0,...,1.881111,False,not_outlier,False,not_outlier,True,high,6,6,6
4135,4136,http://propertyfinder.ae/to/7JD3T0H9T7QNR2S3EQ...,7JD3T0H9T7QNR2S3EQ5Q25X5N4,Dubai,4409,Office Space,Commercial Rent,Yearly,N/A,60000.0,...,1.881111,False,not_outlier,False,not_outlier,True,high,6,6,6
4138,4139,http://propertyfinder.ae/to/70D1CWE9J7FYEEWV9V...,70D1CWE9J7FYEEWV9VCE5R75QG,Sharjah,14475,Land,Residential Sale,Sale,N/A,699000.0,...,1.659211,False,not_outlier,False,not_outlier,True,high,6,6,6
4140,4141,http://propertyfinder.ae/to/01K6D5AF0J54DBZWMS...,01K6D5AF0J54DBZWMSEB67NGXK,Dubai,2448,Office Space,Commercial Rent,Yearly,Studio,5000.0,...,2.108000,False,not_outlier,True,low,True,low,6,6,6


In [68]:
df_copy[df_copy['housing_type_name']=='Bungalow']

,outlier_rank,url,property_listing_id,location_lvl_0_name,location_id,housing_type_name,offering_type_name,property_price_type_name,bedrooms,price,...,sqft_normalized_score,is_pts_outlier,pts_outlier_type,is_price_outlier,price_outlier_type,is_sqft_outlier,sqft_outlier_type,pts_multiplier,price_multiplier,sqft_multiplier


## DTCM and Short Term excluded

In [69]:
# dubai_df

In [70]:
# (dubai_df['complaince_type']=='dtcm') | (dubai_df['listing_short_term_flag'] == True)

In [71]:
# filtered_dubai_df = dubai_df[(dubai_df['complaince_type']!='dtcm') & (dubai_df['listing_short_term_flag'] == False)]
# filtered_dubai_df

In [72]:
# filtered_dubai_df['property_permit_number'].value_counts()

In [73]:
# training_listings = pd.read_parquet(f'../datasets/{version}/training_set_{version}.parquet')
# training_listings

In [74]:
# # filtered_dubai_is_active = (filtered_dubai_df['status']=='Active')
# temp_df = filtered_dubai_df[(filtered_dubai_df['property_permit_number'].isna() | (filtered_dubai_df['property_permit_number'].isin(['0000000000', '1234567' ,'<YOUR_RERA_AD_NUMBER>' ])))]
# temp_df

In [75]:
# temp_df[temp_df['status']=='Active']

In [76]:
# all_listings[all_listings['segment_key']=='39|Apartment|Residential Sale|Sale|2 Bed']

In [77]:
# combined_listings = pd.read_parquet(f'../datasets/{version}/combined_listings_{version}.parquet')

In [78]:
# from configs.outlier_config import HOUSING_TYPE_ID_MAP, OFFERING_TYPE_ID_TO_NAME, PRICE_TYPE_MAP
#
# combined_listings['housing_type_name'] = combined_listings['housing_type_id'].astype(str).replace(HOUSING_TYPE_ID_MAP)
# combined_listings['offering_type_name'] = combined_listings['offering_type_id'].astype(str).replace(OFFERING_TYPE_ID_TO_NAME)
# combined_listings['property_price_type_name'] = combined_listings['property_price_type_id'].astype(str).replace(PRICE_TYPE_MAP)
# segment_key = {
#     'loc': 39,
#     'housing_type': 'Apartment',
#     'sale_type': 'Residential Sale',
#     'price_type': 'Sale',
#     'bedrooms': '2 Bed'
# }
# # combined_listings[(combined_listings['location_id']==segment_key['loc']) & (combined_listings['housing_type_name']==segment_key['housing_type']) & (combined_listings['offering_type_name']==segment_key['sale_type']) & (combined_listings['property_price_type_name']==segment_key['price_type']) & (combined_listings['bedrooms']==segment_key['bedrooms'])]
# training_listings[(training_listings['location_id']==segment_key['loc']) & (training_listings['housing_type_name']==segment_key['housing_type']) & (training_listings['offering_type_name']==segment_key['sale_type'])]

In [79]:
# len(combined_listings)